In [1]:
from src.spark_session import get_spark
from src.utils.display_mimic import display
from src.utils.data_quality import *

spark = get_spark()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b69e3303-48db-4749-a146-4878fba1cd10;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 501ms :: artifacts dl 21ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.0.0 from central in [default]
	io.delta#delta-storage;3.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |  

In [3]:
spark.sql('SHOW TABLES').show(truncate=False)

+---------+----------------------+-----------+
|namespace|tableName             |isTemporary|
+---------+----------------------+-----------+
|default  |raw_product           |false      |
|default  |raw_sales_order_detail|false      |
|default  |raw_sales_order_header|false      |
+---------+----------------------+-----------+



In [5]:
tables = [
    "raw_product",
    "raw_sales_order_header",
    "raw_sales_order_detail"
]

for table in tables:
    print(f"\n===== {table} =====")
    df = spark.table(table)
    df.printSchema()
    display(df)


===== raw_product =====


26/03/05 00:13:59 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/03/05 00:13:59 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist


root
 |-- ProductID: integer (nullable = true)
 |-- ProductDesc: string (nullable = true)
 |-- ProductNumber: string (nullable = true)
 |-- MakeFlag: boolean (nullable = true)
 |-- Color: string (nullable = true)
 |-- SafetyStockLevel: integer (nullable = true)
 |-- ReorderPoint: integer (nullable = true)
 |-- StandardCost: double (nullable = true)
 |-- ListPrice: double (nullable = true)
 |-- Size: string (nullable = true)
 |-- SizeUnitMeasureCode: string (nullable = true)
 |-- Weight: double (nullable = true)
 |-- WeightUnitMeasureCode: string (nullable = true)
 |-- ProductCategoryName: string (nullable = true)
 |-- ProductSubCategoryName: string (nullable = true)




===== raw_sales_order_header =====
root
 |-- SalesOrderID: integer (nullable = true)
 |-- OrderDate: timestamp (nullable = true)
 |-- ShipDate: date (nullable = true)
 |-- OnlineOrderFlag: boolean (nullable = true)
 |-- AccountNumber: string (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- SalesPersonID: integer (nullable = true)
 |-- Freight: double (nullable = true)




===== raw_sales_order_detail =====
root
 |-- SalesOrderID: integer (nullable = true)
 |-- SalesOrderDetailID: integer (nullable = true)
 |-- OrderQty: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- UnitPriceDiscount: double (nullable = true)



In [9]:
print("=== RAW PRODUCT ===")
print("Nulls:", check_not_null(spark, "raw_product", ["ProductID"]))
print("Duplicates:", check_no_duplicates(spark, "raw_product", ["ProductID"]))
print("Negative prices:", check_negative_values(
    spark,
    "raw_product",
    ["ListPrice", "StandardCost"]
))

print("\n=== SALES ORDER DETAIL ===")
print("FK Product:", check_fk_integrity(
    spark,
    "raw_sales_order_detail",
    "raw_product",
    "ProductID"
))


=== RAW PRODUCT ===


Nulls: {'ProductID': 0}
Duplicates: 8


Negative prices: {'ListPrice': 0, 'StandardCost': 0}

=== SALES ORDER DETAIL ===
FK Product: 0


In [13]:
display(spark.table("raw_product") \
     .filter("ProductID IN (SELECT ProductID FROM raw_product GROUP BY ProductID HAVING COUNT(*) > 1)") \
     .orderBy("ProductID"))

,ProductID,ProductDesc,ProductNumber,MakeFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,Size,SizeUnitMeasureCode,Weight,WeightUnitMeasureCode,ProductCategoryName,ProductSubCategoryName
0,713,"Long-Sleeve Logo Jersey, S",LJ-0192-S,False,Multi,4,3,38.4923,49.99,S,None,NaN,None,None,Jerseys
1,713,"Long-Sleeve Logo Jersey, S",LJ-0192-S,False,Multi,4,3,38.4923,49.99,S,None,NaN,None,Clothing,Shirt
2,714,"Long-Sleeve Logo Jersey, M",LJ-0192-M,False,Multi,4,3,38.4923,49.99,M,None,NaN,None,None,Jerseys
3,714,"Long-Sleeve Logo Jersey, M",LJ-0192-M,False,Multi,4,3,38.4923,49.99,M,None,NaN,None,Clothing,Shirt
4,715,"Long-Sleeve Logo Jersey, L",LJ-0192-L,False,Multi,4,3,38.4923,49.99,L,None,NaN,None,None,Jerseys
5,715,"Long-Sleeve Logo Jersey, L",LJ-0192-L,False,Multi,4,3,38.4923,49.99,L,None,NaN,None,Clothing,Shirt
6,716,"Long-Sleeve Logo Jersey, XL",LJ-0192-X,False,Multi,4,3,38.4923,49.99,XL,None,NaN,None,None,Jerseys
7,716,"Long-Sleeve Logo Jersey, XL",LJ-0192-X,False,Multi,4,3,38.4923,49.99,XL,None,NaN,None,Clothing,Shirt
8,881,"Short-Sleeve Classic Jersey, S",SJ-0194-S,False,Yellow,4,3,41.5723,53.99,S,None,NaN,None,None,Jerseys
9,881,"Short-Sleeve Classic Jersey, S",SJ-0194-S,False,Yellow,4,3,41.5723,53.99,S,None,NaN,None,Clothing,Shirt
